2026-04-25
너는 이제부터 관광 일정의 실행 가능성을 정밀하게 검토하는 'QA 엔진 개발자'야.
사용자가 입력한 여행 일정(장소 리스트)을 '수학적 최적해'와 '현실적 휴리스틱' 두 가지 관점에서 분석하는 파이썬 코드를 작성해줘.

[데이터 구조]
- 입력: 장소 이름, 좌표(x, y), 운영시간(open, close), 예상 체류시간이 포함된 JSON 리스트.
- 제약사항: 첫 번째 장소와 마지막 장소는 '숙소'로 고정됨.

[핵심 로직 1: VRPTW 기반 수학적 검증]
- ortools 라이브러리를 사용하여 단일 숙소 기반의 VRPTW(VRP with Time Windows)를 수행해줘.
- 사용자가 입력한 순서의 '총 이동시간'과 알고리즘이 찾은 '최적 순서의 총 이동시간'을 비교해.
- 두 시간의 차이(Gap)를 계산해서 이동 효율성 점수를 산출해줘.

[핵심 로직 2: 휴리스틱 기반 현실성 검증]
- Travel Ratio: (총 이동시간 / 전체 일정시간)이 0.4를 넘으면 감점.
- Total Duration: 전체 일정이 12시간을 초과하면 피로도 감점.
- Opening Hour Risk: 운영 종료 1시간 이내 도착 시 위험 경고.

[출력 형식]
1. 종합 Risk Score (100점 만점)
2. 하위 항목별 점수 (이동 효율성, 시간 현실성, 운영 안전성)
3. 개선 제안: VRPTW가 찾은 최적 순서와 비교하여 "어느 지점에서 낭비가 발생하는지" 설명.
4. 설명 구조: Fact - Rule - Risk - Suggestion 형식을 지킬 것.

최대한 깔끔한 모듈형 코드로 작성해주고, ortools가 설치되지 않은 환경을 고려해 기본적인 Haversine 거리 계산 함수도 포함해줘.

[Claude를 위한 프롬프트 엔지니어링: VRPTW 검증 엔진 구현]

  전문가 역할:
  너는 관광 일정의 수학적 완결성을 검증하는 'VRPTW QA 엔지니어'야. 기존의 데이터 수집 및 분석 로직은
  유지하되, 오직 입력된 일정의 '실행 가능성(Feasibility)'과 '이동 효율성(Efficiency)'을 판단하는
  전용 모듈을 추가하는 것이 목표야.

  1. 핵심 미션:
   - ortools를 사용하여 사용자가 계획한 일정(User Route)을 수학적 최적 경로(Optimal Route)와 대조
     분석해라.
   - 기존 파일을 수정하지 말고 src/validation/vrptw_engine.py를 신규 생성하여 로직을 격리해라.
   - "이 일정은 가능한가?"에 대해 Fact-Rule-Risk-Suggestion 프레임워크로 응답해라.

  2. 입력 데이터 규약 (Input Requirements):
   - 장소 리스트: 이름, 좌표(x, y), 운영시간(open, close), 체류시간(stay_duration).
   - 고정 조건: 2박3일 이상의 일정인 경우에 첫째 날과 마지막 날을 제외하고 하루의 시작의 첫 번째와 마지막 장소는 반드시 '숙소(Depot)'로 고정하여 연산할 것. ex) 2박 3일 일정이라면 1일차에는 반드시 숙소로 귀환, 2일차에는 시작이 숙소이고 마지막 장소도 숙소이고, 마지막 날인 3일차에는 반드시 숙소에서 시작해야 함.
   - 이동 매트릭스: cache_route.json이 존재하면 실제 주행 시간을 사용하고, 없으면 Haversine 거리
     기반(30km/h 가정)으로 자동 전환(Fallback)되게 할 것.

  3. VRPTW 구현 세부 지침:
   - Time Window 제약: 각 장소의 open 시간에 도착 가능해야 하며, close 시간 전에는 반드시 stay_duration을 마치고 떠날 수 있어야 함.
   - 대기 시간(Slack Var) 허용: 너무 일찍 도착할 경우 영업 시작까지 대기하는 시간을 허용하되, 총 일정 시간에 포함시킬 것.
   - 최적화 목표: 최소 이동 시간(Minimize Travel Time)을 목표로 함.

  4. 결과 분석 및 판단 로직 (Judgment Logic): (휴리스틱 이기 때문에 한정량은 변경될 수 있음)
   - Efficiency Gap: (사용자 경로 시간 - 최적 경로 시간) / 최적 경로 시간. 20% 초과 시 '효율성 낮음' 판정.
   - Fatigue Score: 총 일정(첫 출발~마지막 복귀)이 12시간을 초과할 경우 시간당 10점씩 감점.
   - Safety Margin: 운영 종료 60분 이내에 도착하는 일정이 있다면 '위험(Critical)'으로 분류.

  5. 출력 형식 (Output Format):
  모든 분석 결과는 다음 형식을 갖춘 리포트로 출력해야 함:
   1. Validation Summary: 종합 Risk Score (100점 만점) 및 Pass/Fail 판정.
   2. Comparative Analysis: [사용자 순서] vs [VRPTW 최적 순서] 비교표.
   3. Deep Dive (Fact-Rule-Risk-Suggestion): 
      - Fact: 분석된 실제 수치.
  주의사항:
   - 기존의 analysis_result.json 등 분석 결과 파일은 읽기 전용으로만 사용하고 절대 내용을 수정하지
     마라.
   - ortools 미설치 환경에서도 코드 자체가 에러를 내지 않도록 Import 예외 처리를 철저히 해라.

  ---

  [시니어 검토자 의견: Claude에게 전달할 가이드라인]

  Claude가 작업을 시작하기 전에 아래 사항을 명확히 인지시켜야 합니다.

   1. 독립성: VRPTW 모듈은 독립적인 Validator로 작동해야 합니다. 즉, 데이터를 바꾸는 것이 아니라
      "데이터를 보고 점수를 매기는 역할"에 집중해야 합니다.
   2. 타당성 판단의 근거: 단순히 "멀다"가 아니라, "운영 시간(Time Window) 때문에 이 순서로는 마지막 장소 입장이 불가능하다"와 같은 시계열적 근거를 반드시 제시하게 하십시오.
   3. 확장성: 향후 카카오 API를 통해 실제 주행 시간이 들어올 경우를 대비해, TimeMatrix만 교체하면 되는 인터페이스 구조를 유지하도록 지시하십시오. (하지만 계약을 체결하기 어렵기 때문에 기존 구조 및 폴백 구조도 유지해야함)